# 00 · SHAP — one FTTL version, inside its own environment

**Run this notebook once per runnable version, each on that version's kernel — in practice v2 and
v3 (see the env-v1 warning at the end of this cell).** Nothing in it
names a version — it detects which one it is from the running interpreter and reads everything else
from `src/config.py`. So the same file produces the v1, the v2 and the v3 analysis, and the three
outputs are directly stackable later.

**Set `SPLIT` in §0 before running.** The feature matrices exist only per split, and the split
decides what the concentration statistic means — measured on `train` it describes the fitted
function on data it saw, on a holdout it describes generalisation. Split names are each version's
own (`config.SPLITS`); v1 and v2 invert what `test` means and only v3 has `oot`, so they are never
unified. The chosen split lands in the output filename and in the sidecar meta, so no figure can
be reported without it. **`SPLIT` stays a manual, one-at-a-time choice — change it and re-run the
notebook to see another split.**

**§4 computes BOTH SHAP backends in the same run — `interventional` and `tree_path_dependent` —
and §11 saves both.** `ACTIVE_BACKEND` (set in §4) picks which single backend every downstream
plot (§5–§10) draws from; flip it and re-run from §4 to get the identical figure set for the other
backend without recomputing anything from §0–§3. This is automated (unlike `SPLIT`) because it is
cheap — linear in `N_EXPLAIN` — and because §7's interaction values are computed once regardless
(`tree_path_dependent` always, per `shap_kit.py`), so there is no O(rows²) cost to duplicate.

| | v1 | v2 | v3 |
|---|---|---|---|
| kernel / interpreter | `src/envs/v1/.venv` | `src/envs/v2/.venv` | `src/envs/v3/.venv` |
| xgboost | **0.72** | **1.4.2** | **3.2.0** |
| decision rule | segmented on mobility (0.75 / 0.85) | one global cutoff, changed over time | one global cutoff (0.984) |

Those three xgboost releases are seven years apart, so the notebook never touches `shap`'s plotting
layer (`shap.plots.beeswarm` and the `Explanation` object do not exist in the 2018-era stack). Every
figure is drawn from the raw φ matrix by `src/shap_kit.py` with plain matplotlib. If `shap` is
importable in the env it is used for the values; if not, the booster's own `pred_contribs` is —
exact TreeSHAP either way, and §4 records which.

**Setting up a kernel for a version env** (once per env, on the company laptop):

```bash
src/envs/v2/.venv/bin/python -m pip install ipykernel matplotlib pandas pyarrow
src/envs/v2/.venv/bin/python -m ipykernel install --user --name fttl-v2 --display-name "FTTL v2 (env-v2)"
```

What it produces, per version and split: the figures in `figures/`, and, per backend,
`src/data/real/detection/shap/<v>/<v>_attributions_<split>.parquet` (the `ACTIVE_BACKEND` one — the
canonical name `src/scoring/attribute.py` also writes) plus
`<v>_attributions_<split>_<other_backend>.parquet` (the sibling, e.g.
`..._tree_path_dependent.parquet`) — each with its own `_meta.json`. The cross-version comparison
in `00_shap_attribution.ipynb` reads only the canonical, no-suffix file, so the same artefact
`src/scoring/attribute.py` writes headlessly can be read without anything being recomputed.

**Figure naming convention.** Early figures (`00`…`09` before the `VERSION`/`ACTIVE_BACKEND`
prefix, e.g. `_00_score_distribution`, `_07_waterfall_…`, `_09_band_bars`) are a flat counter
assigned in writing order and are **frozen — never renumbered**, so an existing figure keeps the
same filename (and thesis reference) no matter what gets added around it later. **Any figure added
to this notebook from now on is named after its own section instead: `<section>_<NN>`**, e.g. a
second figure added to §7 is `07_02`, a first one added to §10 is `10_01`. That is always two
numeric groups where a frozen name has one, so the two schemes cannot collide even when a section
number matches an already-used flat counter (§9b's `09_01_near_tau_band_comparison` sits fine next
to §10's frozen `09_band_bars` — one token vs two). This keeps the notebook safe to keep extending
without ever touching a name a figure or the thesis already depends on.

> ⚠️ **env-v1 cannot run this notebook, and cannot run `scoring/attribute.py` either.** Three
> independent blockers, all confirmed: (a) `config.py` and `shap_kit.py` use Python-3.6+/3.7+
> syntax (f-strings, `from __future__ import annotations`, `str | None`) and env-v1 is Python
> 3.5.6; (b) env-v1 has **no parquet engine at all** — `src/envs/v1/requirements.txt` carries
> neither pyarrow nor fastparquet, which is why `01_export_v1.ipynb` writes CSV; (c) `shap` on
> Python 3.5 is capped at the 2018 releases. So **v1 has no φ yet**, and the cross-version
> notebook will report v1 as missing until a 3.5-compatible extractor exists. Producing one
> means backporting `attribute.py` (syntax **and** CSV I/O) — `training/retrain.py` is the
> worked example of both.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config # real paths, columns, rules, hyperparameters
import shap_kit as sk               # noqa: E402  — everything version-agnostic lives here

sk.style()

# Which version is this kernel? Inferred from the interpreter path (src/envs/v2/.venv/...).
# Override ONLY if the env lives somewhere else; a wrong value here silently analyses the wrong
# model, which is why env_report() then checks the xgboost release against config.
VERSION = sk.detect_version()
ENV = sk.env_report(VERSION, strict=True)
VERSION = ENV["version"]

SOURCE      = "real"
# WHICH SPLIT. features/targets/scores exist only per split — the export notebooks write one
# file each and no unsplit base file — so this is not a default anyone can skip. It also decides
# what the concentration number MEANS: on "train" it describes the fitted function on data it
# saw; on a holdout it describes generalisation. Valid names are that version's own, and they
# differ (v1: train/test/val1/val2 · v2: train/val/test · v3: train/test/oot).
SPLIT       = config.SPLITS[VERSION][0]    # <<< set deliberately; [0] is that version's "train"
#            config.SPLITS[VERSION] is a TUPLE, so it is indexed by POSITION, never by name.
#            To take the holdout instead: SPLIT = config.OOT_SPLIT[VERSION]  (v1 val2 · v2 test · v3 oot)
#            Splits stay MANUAL — pick one, run the notebook, change SPLIT, run it again. §4
#            below only automates the OTHER axis (interventional vs tree_path_dependent), which
#            is cheap (linear in N_EXPLAIN); splits are not swept because re-deriving X_all per
#            split means re-loading/re-preprocessing a whole matrix, and interaction values in §7
#            are O(rows²) — a split sweep would multiply that, a backend split does not (§7's
#            interaction values are tree_path_dependent only regardless of ACTIVE_BACKEND, so
#            they are computed once, not per backend).
N_EXPLAIN   = 5000     # claims to attribute (SHAP cost is linear in this)
N_BACKGROUND = 500     # interventional reference sample SIZE. §4 always computes BOTH backends
                        # side by side now (interventional against this background, and
                        # tree_path_dependent against each tree's own cover) — this only sizes
                        # the former; it is no longer a None-to-switch-backend toggle.
SEED        = 0
assert SPLIT in config.SPLITS[VERSION], (SPLIT, config.SPLITS[VERSION])
assert N_BACKGROUND, ("N_BACKGROUND must be a positive int — §4 needs it to build the "
                       "interventional background. tree_path_dependent is computed alongside it "
                       "unconditionally, so this is no longer how you opt into that backend.")
print(f"\nanalysing {VERSION} · split {SPLIT}")

## 1 · What this model actually is

Printed before anything else, and from `config` rather than from memory, because every figure below
has to be read against it. Two facts that matter more than usual here:

- **v2 regularises with penalties** (`reg_alpha=20`, `gamma=15`, `max_depth=10`) while **v3
  regularises with tree structure** (`max_depth=3`, `max_leaves=18`, `min_child_weight=44`). L1
  concentrates feature importance *by construction*, so a flatter or sharper SHAP ranking is partly
  a configuration fact, not only a data fact (`problem.md` §1.4c).
- **v1 sets almost nothing** — so most of its configuration is xgboost **0.72's** defaults, which
  are not the defaults a modern reader assumes.

`scale_pos_weight` (4.5 in v2, 5.55 in v3) also means the score is **not** a calibrated probability;
it is a ranking whose cutoff was chosen to hold precision ≥ 0.985.

In [ ]:
print(f"--- {VERSION} training configuration (config.TRAINING_CONFIG) ---")
cfg = sk.training_config(VERSION)
display(cfg.to_frame())

rule = config.DECISION_RULES[VERSION]
print(f"\n--- decision rule: {rule['shape']} ---")
for k, v in rule.items():
    if not k.startswith("_"):
        print(f"  {k}: {v}")

## 2 · The feature space of *this* repo

Feature count and character are a per-version fact and one of the findings in their own right: the
versions do not merely re-weight a shared feature set, they have different ones (v2's top feature
`location_Home` does not exist in v1 or v3 at all). No cross-version merging happens here — names
stay exactly as the model has them.

The matrix is the **post-preprocessing** one, i.e. what the estimator actually consumes. If the
version repo ships it, `config` resolves it; otherwise the fitted preprocessor pickle is loaded and
applied to the raw claim table — both routes are handled below, and it says which one it took.

In [ ]:
# --- the model -----------------------------------------------------------------------
MODEL_PATH = None        # override only if config's declaration is not what you want to explain
model_path = Path(MODEL_PATH) if MODEL_PATH else config.path("model", VERSION, SOURCE)
print(f"model: {model_path}")
est = sk.load_estimator(model_path)
print(f"  {type(est).__name__}")

# --- the feature matrix --------------------------------------------------------------
FEATURES_PATH = None     # override to point at a specific file
ID_COL = "claim_id"      # canonical name; the version's own name comes from config.column()

def read_table(path):
    """Dispatch on suffix. The real repos' data artefacts are pandas PICKLES on the Z: drive —
    readable precisely because this notebook runs inside that version's own env."""
    path = str(path)
    if path.endswith(".pkl"):
        return pd.read_pickle(path)
    if path.endswith(".csv"):     # env-v1's 2018-era pandas may not have a working pyarrow
        return pd.read_csv(path)
    return pd.read_parquet(path)

def id_column_of(df):
    "The claim id column present in df: canonical first, then this version's own name."
    if ID_COL in df.columns:
        return ID_COL
    try:
        real = config.column(VERSION, "claim_id")
        return real if real in df.columns else None
    except (KeyError, ValueError):    # placeholder / declared-absent
        return None

def load_matrix():
    "Post-preprocessing X, from the declared file if it exists, else via the preprocessor pkl."
    path = (Path(FEATURES_PATH) if FEATURES_PATH
            else config.path("processed_inputs", VERSION, SOURCE, split=SPLIT))
    if path.exists():
        print(f"processed_inputs: {path}  (pre-built matrix)")
        return read_table(path), f"processed_inputs file ({path.suffix.lstrip('.')})"
    print(f"processed_inputs: {path} does not exist -> falling back to preprocessor + raw_dataset")
    prep = sk.load_estimator(config.path("preprocessor", VERSION, SOURCE))
    raw_path = config.path("raw_dataset", VERSION, SOURCE)
    raw = read_table(raw_path)
    matrix = prep.transform(raw)
    names = sk.model_feature_names(est)
    if not names:
        getter = getattr(prep, "get_feature_names_out", None)
        names = list(getter()) if getter is not None else []
    if not names:
        raise RuntimeError(
            "neither the estimator nor the preprocessor exposes post-preprocessing feature "
            "names — a positional matrix would make every per-feature claim unverifiable. "
            "Point FEATURES_PATH at a named matrix instead."
        )
    out = pd.DataFrame(np.asarray(matrix), columns=list(names))
    id_col = id_column_of(raw)
    if id_col:      # sklearn transform preserves row order, so positional carry-over is safe
        out.insert(0, ID_COL, raw[id_col].values)
    return out, f"preprocessor applied to {raw_path.name}"

frame, provenance = load_matrix()

# resolve the id column (canonical, else this version's own name renamed to canonical)
id_col = id_column_of(frame)
if id_col and id_col != ID_COL:
    frame = frame.rename(columns={id_col: ID_COL})
if ID_COL in frame.columns:
    ids = frame[ID_COL]
else:
    # No claim id anywhere. Positional ids are fine WITHIN this run, but the saved attributions
    # cannot be joined to the log / other artefacts — say so loudly rather than silently.
    print("⚠ no claim id column found — using positional ids. The saved parquet will NOT join "
          "to the log; rebuild the matrix with the id carried through before using it further.")
    ids = pd.Series(np.arange(len(frame)), name=ID_COL)

# The real transformed tables carry NON-feature columns too (v1's inputs_transformed.pkl holds
# the target and claimnumber alongside the 37 inputs). Keep exactly what the booster was trained
# on, and SAY what was set aside — dropping silently is how a wrong matrix reaches a figure.
X_all = frame.drop(columns=[ID_COL]) if ID_COL in frame.columns else frame
trained = sk.model_feature_names(est)
if trained:
    extra = [c for c in X_all.columns if c not in trained]
    if extra:
        print(f"set aside {len(extra)} non-model column(s): {extra[:8]}"
              + (" …" if len(extra) > 8 else ""))
    X_all = X_all[[c for c in X_all.columns if c in trained]]
else:
    # The booster carries no feature names, so it cannot say which columns it consumed. Falling
    # back to "everything except claim_id" would feed the model ITS OWN TARGET: the matrix
    # carries it in every version (v2 writes `[ID] + MODEL_FEATURES + [TARGET]`; v1 and v3 write
    # the whole transformed frame, and v3's also holds fttl_predicted_prob / fttl_predicted_label).
    #
    # The registry is the same authority as the booster, just recorded on disk: it is written by
    # features/extract_features.py from this version's own pickles, inside this same env.
    reg_path = config.registry_path(VERSION)
    if not reg_path.exists():
        raise RuntimeError(
            f"the estimator exposes no trained feature names and {reg_path} does not exist, so "
            f"which columns of the matrix are model inputs cannot be established. Build it in "
            f"THIS env first:\n"
            f"    <this python> features/extract_features.py --version {VERSION}")
    registry = json.loads(reg_path.read_text(encoding="utf-8"))
    trained = [str(c) for c in (registry.get("model_features") or [])]
    if not trained:
        raise RuntimeError(
            f"{reg_path} has no `model_features` — extract_features.py could not recover them "
            f"from the pickle. Resolve that before reading any per-feature claim.")
    missing = [c for c in trained if c not in X_all.columns]
    if missing:
        raise RuntimeError(f"the registry names {len(missing)} column(s) absent from the matrix, "
                           f"e.g. {missing[:8]} — stale registry, or the wrong matrix.")
    extra = [c for c in X_all.columns if c not in trained]
    if extra:
        print(f"set aside {len(extra)} non-model column(s) per the registry: {extra[:8]}"
              + (" …" if len(extra) > 8 else ""))
    print(f"  (column list from {reg_path.name}, source "
          f"{registry.get('model_features_source', 'unrecorded')})")
    X_all = X_all[trained]
X_all = sk.align(X_all, est)
non_float = X_all.dtypes[X_all.dtypes != "float64"]
if len(non_float):
    print(f"casting {len(non_float)} non-float64 column(s) to float64 for shap: "
          f"{list(non_float.index[:8])}" + (" …" if len(non_float) > 8 else ""))
    X_all = X_all.astype("float64")

print(f"\n--- {VERSION} feature space ({provenance}) ---")
display(sk.feature_summary(X_all).to_frame(VERSION))

In [ ]:
profile = sk.describe_features(X_all)
print("per-feature profile — sorted by cardinality (full table in `profile`)")
display(profile.sort_values("n_unique", ascending=False).head(25))

fams = profile[profile["kind"] == "binary"]["family"].value_counts()
if (fams > 1).any():
    print("\none-hot families (raw columns that share a prefix) — NOT collapsed, only counted:")
    display(fams[fams > 1].head(15).to_frame("n_columns"))

## 3 · Scores and the fast-track region

Scored live with the loaded model, so this is the exact function the SHAP values below decompose.
The cutoff comes from `config.DECISION_RULES` and differs in *shape* per version — v1 has two
cutoffs keyed on vehicle mobility, v2's single cutoff moved in time, v3's is fixed. A scalar `TAU`
is derived for plotting, and what was assumed is printed. **v2's TAU is currently HARDCODED to
0.872** rather than derived from the regime table — a deliberate simplification, not the exact
per-row rule (v2's cutoff actually alternates 0.8915 / 0.872 / 0.825 across 5 regimes over time;
see `src/threshold.py:apply` for the exact version to switch to later).

In [ ]:
scores = est.predict_proba(X_all)[:, 1]

if rule["shape"] == "global":
    TAU = rule["threshold"]
    tau_note = "single global cutoff"
elif rule["shape"] == "piecewise_global":
    # HARDCODED for now, not derived from rule["regimes"]. v2's cutoff actually moved 4x
    # (0.8915 -> 0.872 -> 0.825 -> 0.872 -> 0.825) and does NOT reduce to one number — but 0.872
    # is the value in force across most of the window this notebook currently explains, so it is
    # used as a stand-in. TODO: swap for src/threshold.py:apply (exact, per-row by date) once the
    # date column's availability in this matrix is confirmed on the company laptop.
    TAU = 0.872
    _hist = " -> ".join(str(r["threshold"]) for r in rule["regimes"])
    tau_note = (f"piecewise in time ({_hist}) — HARDCODED to {TAU} as a stand-in; rows decided "
                f"under an earlier/later regime are mislabelled by this shortcut, so treat the "
                f"split as approximate until threshold.apply() replaces it")
else:                                   # v1: segmented on mobility
    TAU = max(rule["thresholds"].values())
    tau_note = (f"segmented on mobility {rule['thresholds']} — using the HIGHER cutoff ({TAU}), so "
                f"'above τ' here is the set scrapped under EITHER segment; the band "
                f"{rule['overlap_band']} is where mobility decides")

above = scores > TAU        # STRICT, matching src/threshold.py — a car scoring exactly tau is garaged
print(f"τ = {TAU}   ({tau_note})")
print(f"scored {len(scores)} claims · above τ: {above.sum()} ({above.mean():.2%})")
print(f"score range {scores.min():.4f} – {scores.max():.4f}, median {np.median(scores):.4f}")

fig, ax = plt.subplots(figsize=(sk.FIG_W, 3.2))
ax.hist(scores, bins=80, color=sk.BLUE)
ax.axvline(TAU, color=sk.RED, lw=1.4, ls="--")
ax.text(TAU, ax.get_ylim()[1] * 0.95, f" τ = {TAU}", color=sk.RED, fontsize=9, va="top")
ax.set_yscale("log")
ax.set_xlabel(f"model_{VERSION}_score")
ax.set_ylabel("claims (log)")
ax.set_title(f"{VERSION} — score distribution and the fast-track cutoff")
fig.tight_layout()
if hasattr(sk, "figstyle") and sk.figstyle:
    sk.figstyle.save(fig, f"{VERSION}_03a_score_distribution")
plt.show()

## 4 · SHAP values — both backends, side by side

A seeded sample of `N_EXPLAIN` claims is explained twice: once **interventional** (measured
against the `N_BACKGROUND` reference sample below — "versus these reference claims") and once
**tree_path_dependent** (measured against each tree's own cover statistics — "versus this
model's training distribution"). Both are exact TreeSHAP; they are not the same quantity, and
this cell computes and saves both rather than picking one, so a claim about "the backend" is
never silently whichever one happened to run.

`ACTIVE_BACKEND` then decides which single backend every plot from here through §10 draws from
— duplicating every one of those plots for both backends would multiply this whole notebook's
runtime and figure count, which is not what was asked for; instead flip `ACTIVE_BACKEND` and
re-run from §4 to get the identical figure set for the other backend. §11 saves **both**
regardless of which one is active.

`check_additivity` is run for each: φ must sum to the model's own raw margin. A gap of order
1e-6 means the matrix, the column order and the estimator all line up; a gap of order 1 means
they do not, and every figure below would be quietly wrong.

In [ ]:
rng = np.random.RandomState(SEED)
n = len(X_all)
exp_idx = rng.choice(n, size=min(N_EXPLAIN, n), replace=False)
X = X_all.iloc[exp_idx]
s = scores[exp_idx]
row_ids = ids.iloc[exp_idx].values

bg_idx = rng.choice(n, size=min(N_BACKGROUND, n), replace=False)
background = X_all.iloc[bg_idx]

# Same X, same seed — the only thing that differs between the two entries below is the
# reference distribution `sk.compute` measures against (shap_kit.py: passing `background`
# switches feature_perturbation to "interventional"; passing None leaves it tree_path_dependent).
BACKENDS = {"interventional": background, "tree_path_dependent": None}
att_by_backend = {}
for name, bg in BACKENDS.items():
    a = sk.compute(est, X, background=bg, backend="shap")
    sk.check_additivity(a, est)
    assert a.perturbation == name, (
        f"requested {name}, got {a.backend}/{a.perturbation}")
    print(f"[{name}] {a}")
    att_by_backend[name] = a

assert not np.array_equal(att_by_backend["interventional"].phi,
                          att_by_backend["tree_path_dependent"].phi), \
    "both backends produced IDENTICAL phi — one of them silently fell back"

# What every single-backend plot below (§5, §6, §6b, §8, §9, §10) reads. Both backends are
# already computed and both are saved in §11 no matter what this is set to — flip it and re-run
# from here to reproduce the same figures for the other backend.
ACTIVE_BACKEND = "interventional"
att = att_by_backend[ACTIVE_BACKEND]

cmp = pd.concat({name: a.mean_abs for name, a in att_by_backend.items()}, axis=1)
print(f"\ntop features by mean|SHAP|, both backends side by side ({VERSION} · {SPLIT}):")
display(cmp.sort_values(ACTIVE_BACKEND, ascending=False).head(15).round(5))

### 4b · Backend comparison

The one plot that always shows both backends, regardless of `ACTIVE_BACKEND`: the same top
features' mean|SHAP| under each. A ranking that survives the switch is a claim about the model;
one that flips is partly a claim about which reference distribution was chosen.

In [ ]:
feats = cmp.sort_values(ACTIVE_BACKEND, ascending=False).head(15).index[::-1]  # smallest->largest, matches plot_bar
colors = {"interventional": sk.BLUE, "tree_path_dependent": sk.RED}

fig, ax = plt.subplots(figsize=(sk.FIG_W, 0.32 * len(feats) + 1.4))
y = np.arange(len(feats))
h = 0.36
for i, name in enumerate(BACKENDS):
    offset = h / 2 if i == 0 else -h / 2
    ax.barh(y + offset, cmp.loc[feats, name].values, height=h, color=colors[name], label=name)
ax.set_yticks(y)
ax.set_yticklabels(feats, fontsize=8)
ax.set_ylim(-0.7, len(feats) - 0.3)
ax.set_xlabel("mean |SHAP|  (log-odds)")
ax.set_title(f"{VERSION} · {SPLIT} — global importance, interventional vs tree_path_dependent")
ax.legend(fontsize=8)
fig.tight_layout()
if sk.figstyle:
    sk.figstyle.save(fig, f"{VERSION}_04a_shap_bar_backend_comparison")
plt.show()

## 5 · Input distributions, below vs above the cutoff

Shown for the features the model actually leans on (top by mean |SHAP|), because that is where a
distribution difference has consequences. A tree model with a hard threshold acts on a *region* of
input space: the above-τ population is where labels get forced, so how it differs on each driver is
the descriptive half of the SFP story. Binary/one-hot columns are drawn as rates, continuous ones as
overlaid histograms.

In [ ]:
DIST_FEATURES = att.top(9)
fig = sk.plot_distributions(X, DIST_FEATURES, scores=s, tau=TAU, ncols=3,
                            title=f"{VERSION} — input distributions of the top SHAP drivers "
                                  f"({ACTIVE_BACKEND})")
if sk.figstyle:
    sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_05a_input_distributions")
plt.show()

## 6 · Global interpretability

Three views of the same φ matrix, and they are not redundant:

1. **Bar — mean |SHAP|.** The ranking. One number per feature; says how much, never which way.
2. **Beeswarm — signed.** One point per claim, coloured by the feature's value. This is where
   *direction* appears: red on the right = high values push towards total loss. Red on **both**
   sides means the effect is conditional, which §7's dependence plots then chase.
3. **Beeswarm — magnitude, coloured by mean |SHAP|.** The same rows folded to |φ|. It separates a
   feature that is decisive for a small subset from one that matters mildly everywhere — the two
   are indistinguishable in the bar chart, and they mean very different things for a fast-track
   policy that only ever acts on the tail.

In [ ]:
fig = sk.plot_bar(att, top_n=20, title=f"{VERSION} — global importance (mean |SHAP|, {ACTIVE_BACKEND})")
if sk.figstyle:
    sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_06a_shap_bar")
plt.show()

In [ ]:
fig = sk.plot_beeswarm(att, top_n=20, title=f"{VERSION} — SHAP beeswarm (signed, {ACTIVE_BACKEND})")
if sk.figstyle:
    sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_06b_shap_beeswarm")
plt.show()

In [ ]:
fig = sk.plot_beeswarm_abs(att, top_n=20,
                           title=f"{VERSION} — |SHAP| spread, coloured by mean |SHAP| ({ACTIVE_BACKEND})")
if sk.figstyle:
    sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_06c_shap_beeswarm_abs")
plt.show()

## 6b · shap's own plots — where the env allows it (v2 / v3)

Everything above is drawn from the raw φ matrix by `shap_kit` with plain matplotlib, because this
notebook also has to run under **env-v1: Python 3.5, shap 0.28 (2018), xgboost 0.72**. The
`Explanation` object and the whole `shap.plots.*` namespace arrived in **shap 0.36 (Sept 2020)** —
they simply do not exist there.

That constraint binds v1 **only**. env-v2 (py3.10) and env-v3 (py3.11) can both carry a current
shap, so this section gives them shap's native rendering. It **recomputes nothing**:
`sk.explanation(att)` wraps the φ, base values and feature matrix we already have, so these plots
and the ones above are the same numbers drawn by different code.

⚠️ **Do not put these beside another version's figure.** A cross-version comparison must be drawn
by identical code for every version, or part of the difference between two panels is a difference
between two renderers. §6's plots are the comparable ones; §6b is for reading *this* version
closely. The cell degrades to a printed explanation on v1 rather than failing.


In [ ]:
cap = sk.shap_capability()
print(f"shap in this env: {cap}")

if not cap["plots"]:
    print(f"\n[{VERSION}] shap.plots is unavailable here — this is expected on env-v1 "
          f"(Python 3.5 caps shap at the 2018 releases, which predate Explanation).")
    print("§6 above already drew the same phi with plain matplotlib. Nothing is missing from the")
    print("analysis; only shap's own rendering of it is.")
else:
    expl = sk.explanation(att)          # wraps the phi already computed in §4 — no second pass
    print(f"\nExplanation ({ACTIVE_BACKEND}): values{expl.values.shape}  base{np.shape(expl.base_values)}  "
          f"{len(expl.feature_names)} features")

    fig = sk.native_plot("beeswarm", expl, max_display=20)
    if sk.figstyle:
        sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_06d_shap_native_beeswarm")
    plt.show()

    fig = sk.native_plot("bar", expl, max_display=20)
    if sk.figstyle:
        sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_06e_shap_native_bar")
    plt.show()

    # One claim, decomposed. The waterfall is the clearest picture of the additivity identity:
    # base_value + sum(phi) = this claim's raw margin.
    top_row = int(np.argmax(s))          # the highest-scoring claim in the explained sample
    fig = sk.native_plot("waterfall", expl[top_row], max_display=14)
    if sk.figstyle:
        sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_06f_shap_native_waterfall")
    plt.show()
    print(f"waterfall: highest-scoring explained claim, score {s[top_row]:.4f} "
          f"({'above' if s[top_row] > TAU else 'below'} tau={TAU})")


## 7 · Interactions — what the vertical dispersion is

> *"Vertical dispersion in SHAP values seen for fixed variable values is due to interaction effects
> with other features in our FTTL system."*

**What that sentence claims.** Take a dependence plot and stand on one value of x — say every claim
with `repair_ratio = 0.8`. All of them handed the model the *same* value of that feature. If the
feature acted on its own, they would all receive the *same* contribution φ and the plot would be a
thin curve. It is not: at that single x the points spread over a vertical band. The model gave the
same input value different credit in different claims, and the only thing that differs between them
is **the rest of the claim**. That is what an interaction is: the worth of `repair_ratio = 0.8`
depends on the car's age, its value, whether the airbag deployed.

Mechanically, inside a tree, it is claims taking different paths — a split on another feature higher
up puts the `repair_ratio` split in a different context, so the same split earns a different amount.

So the reading is: **height of the band = how much this feature acts alone; thickness of the band =
how much it acts jointly with something else.** Colouring by the right second feature turns that
thickness from noise into a pattern.

**Why it matters here specifically.** Under a hard cutoff the fast-tracked claims are a *region* of
input space, not a slice of one variable. If a driver's contribution is conditional on a second
feature, the population above τ is selected on the **pair** — and the forced labels that flow into
the next model carry that pairing with them. A thick band, organised by a second feature, is that
selection made visible at the feature level.

Three routes to it, and they answer **different** questions:

| | what it measures | cost |
|---|---|---|
| `interaction_values` → `interaction_strength` | what the **model** does jointly with a pair (exact TreeSHAP) | O(rows × p²) — the expensive one |
| `feature_association` (pearson / spearman / mutual_info) | whether the two **columns** are related in the data | cheap |
| Friedman's H-statistic | non-additivity of the partial-dependence surface | many model calls; not implemented — TreeSHAP is exact for trees |

They disagree routinely. Two highly correlated features often show **no** interaction (the trees read
one and ignore the other), and two independent features can interact strongly. Reporting a
correlation as an interaction is a real error, so the two are computed separately below and shown
side by side.

In [ ]:
# Interaction values cost O(rows × p²) in memory, so ROWS are the lever: a few hundred is plenty
# to RANK pairs, which is all that is needed to choose what to colour and plot. Anything reported
# as a magnitude should come from the full φ matrix above, not from this subsample.
# X must keep EVERY model feature — the model cannot predict on a column subset, so restricting
# features is done downstream, on the returned strength matrix, never on the input.
#
# Computed ONCE, not per backend: interaction_values is tree_path_dependent regardless of
# ACTIVE_BACKEND (shap_kit.py: "no background here on purpose"), so re-running it under the
# other backend would recompute the identical numbers at O(rows²) cost for nothing.
INTER_ROWS = 300

try:
    inter, X_inter, inter_backend = sk.interaction_values(est, X, max_rows=INTER_ROWS, seed=SEED)
    strength = sk.interaction_strength(inter, X_inter.columns)
    print(f"backend: {inter_backend}")
    display(sk.top_interactions(strength, k=12).round(4))
except (MemoryError, RuntimeError) as exc:
    strength = None
    print(f"interaction values unavailable -> {exc}")
    print("\nThe dependence plots below fall back to the correlation proxy for their colour.")

In [ ]:
if strength is not None:
    fig = sk.plot_interaction_heatmap(strength, top_n=15,
                                      title=f"{VERSION} — pairwise SHAP interaction strength")
    if sk.figstyle:
        sk.figstyle.save(fig, f"{VERSION}_07a_interaction_heatmap")
    plt.show()

In [ ]:
# The DATA-side question, for contrast: which columns are merely related to each other?
assoc = sk.feature_association(X, method="spearman", features=att.top(30))
print("strongest column associations in X (Spearman) — a property of the data, NOT of the model:")
display(sk.top_associated_pairs(assoc, k=10).round(3))

if strength is not None:
    pairs = sk.top_interactions(strength, k=10)
    pairs["spearman_in_X"] = [
        float(assoc.loc[a, b]) if (a in assoc.index and b in assoc.columns) else np.nan
        for a, b in zip(pairs.feature_a, pairs.feature_b)]
    print("\nthe model's strongest interactions with the data association beside them. A strong "
          "interaction at near-zero correlation is a genuinely joint effect — not two columns "
          "carrying the same information:")
    display(pairs[["feature_a", "feature_b", "interaction", "spearman_in_X"]].round(4))

## 8 · Dependence — the value → contribution shape

x is the feature's value, y is its contribution to the log-odds, and the colour is the feature that
**actually** interacts with it most — taken from §7's matrix rather than from a correlation proxy.
Read the vertical thickness first, then whether the colour explains it: a band that separates into
red-above / blue-below is the interaction resolved.

The **step** structure along x is the tree's split points. A step sitting where the fast-track
population begins is the model having learned the decision boundary rather than the physics of the
damage — which is exactly what an SFP loop would produce.

In [ ]:
DEP_FEATURES = [f for f in att.top(12)
                if pd.api.types.is_numeric_dtype(X[f]) and X[f].nunique() > 4][:4] or att.top(3)
for feat in DEP_FEATURES:
    partner = sk.pick_interaction(strength, feat) if strength is not None else None
    fig = sk.plot_dependence(att, feat, interaction="auto", strength=strength,
                             title=f"{VERSION} — dependence: {feat}  ({ACTIVE_BACKEND})" +
                                   (f"   (coloured by {partner})" if partner else ""))
    if sk.figstyle:
        sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_08a_dependence_{feat}")
    plt.show()

In [ ]:
# The strongest interacting PAIRS, plotted as pairs — one feature's dependence coloured by the
# specific partner it interacts with. This is the plot the interaction matrix exists to choose.
if strength is not None:
    for _, row in sk.top_interactions(strength, k=3).iterrows():
        a, b = row.feature_a, row.feature_b
        fig = sk.plot_dependence(att, a, interaction=b,
                                 title=f"{VERSION} — {a} × {b}  (interaction {row.interaction:.3f}, {ACTIVE_BACKEND})")
        if sk.figstyle:
            sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_08b_pair_{a}__{b}")
        plt.show()

## 9 · Local interpretability — individual claims

Three claims chosen to bracket the decision, not at random:

- **the strongest fast-track** (highest score) — the model at its most certain,
- **just above τ** — a car scrapped without inspection on a hair's margin,
- **just below τ** — its near-twin, which went to a garage and got an independent outcome.

The last two are the pair the whole SFP argument turns on: nearly identical inputs, opposite
treatment, and only one of them ever produces a verifiable label. The waterfall gives the full exact
decomposition; the force plot compresses it so several claims can be compared at a glance.

In [ ]:
order = np.argsort(s)
picks = {}
if above[exp_idx].any():
    picks["strongest fast-track"] = int(order[-1])
    just_above = np.where(s[order] > TAU)[0]
    if len(just_above):
        picks["just above τ"] = int(order[just_above[0]])
below = np.where(s[order] < TAU)[0]
if len(below):
    picks["just below τ"] = int(order[below[-1]])

for label, row in picks.items():
    print(f"{label}: row {row}  ·  claim {row_ids[row]}  ·  score {s[row]:.4f}")
    fig = sk.plot_waterfall(att, row=row, top_n=12,
                            label=f"{VERSION} — {label}  (score {s[row]:.4f}, {ACTIVE_BACKEND})")
    if sk.figstyle:
        sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_09a_waterfall_{label.replace(' ', '_')}")
    plt.show()

In [ ]:
for label, row in picks.items():
    fig = sk.plot_force(att, row=row, top_n=10,
                        label=f"{VERSION} — force: {label}  (score {s[row]:.4f}, {ACTIVE_BACKEND})")
    if sk.figstyle:
        sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_09b_force_{label.replace(' ', '_')}")
    plt.show()

### 9b · Averaged over many claims — the band immediately around τ

The single "just above / just below" pair above is anecdotal — one claim on each side. Widening it
to the `N_NEAR_ABOVE` claims scoring closest to τ from above and the `N_NEAR_BELOW` claims closest
from below turns the same argument into a population comparison: does the feature that decided one
claim's waterfall hold up as the average reason across the whole boundary, or was it a one-off?

The two counts are independent parameters, not one shared n — the score distribution is not
symmetric around τ (scores bunch differently on each side of a fast-track cutoff), so a fixed n on
both sides would not be a like-for-like sample.

In [ ]:
# How many claims on each side of τ to average the SHAP band comparison over. Kept as two
# independent parameters (not one shared n) because the two sides are not a like-for-like sample.
N_NEAR_ABOVE = 50    # claims scoring closest to τ from ABOVE (nearest the boundary first)
N_NEAR_BELOW = 50    # claims scoring closest to τ from BELOW

above_mask = above[exp_idx]     # STRICT s > TAU, same convention as §3/§9
below_mask = ~above_mask

near_above_idx = np.where(above_mask)[0][np.argsort(s[above_mask])[:N_NEAR_ABOVE]]
near_below_idx = np.where(below_mask)[0][np.argsort(-s[below_mask])[:N_NEAR_BELOW]]

near_labels = np.full(len(s), np.nan, dtype=object)
near_labels[near_below_idx] = "below τ"
near_labels[near_above_idx] = "above τ"
near_bands = pd.Series(pd.Categorical(near_labels, categories=["below τ", "above τ"], ordered=True))

print(f"near-cutoff band: {len(near_below_idx)} claims below τ (closest) vs "
      f"{len(near_above_idx)} claims above τ (closest) — requested {N_NEAR_BELOW}/{N_NEAR_ABOVE}")

fig, near_table = sk.plot_band_bars(
    att, near_bands, top_n=12,
    title=f"{VERSION} — mean |SHAP|, {len(near_below_idx)} closest-below vs "
          f"{len(near_above_idx)} closest-above τ ({ACTIVE_BACKEND})")
if sk.figstyle:
    sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_09c_near_tau_band_comparison")
plt.show()
display(near_table.round(4))

### 9c · The same claims, kept individual — not averaged

§9b collapsed the `N_NEAR_ABOVE` / `N_NEAR_BELOW` claims from the cell above into one mean per
feature; that hides a feature that only matters for a handful of them inside the average. This
keeps every one of those same claims as its own row — no aggregation — sorted by score
(descending) within each side, and drawn as two separate panels so the above-τ and below-τ
populations can be read side by side rather than pooled into one figure.

In [ ]:
# Same claim sets as §9b (near_above_idx / near_below_idx), no new sampling. Columns are the
# top features by mean|SHAP| (att.top — same ranking §6's bar chart uses); rows are those claims,
# sorted by SCORE DESCENDING within each side so the deepest-scoring claim on each panel is first.
TOP_N_FEATURES_9C = 12

feats_9c = att.top(TOP_N_FEATURES_9C)
feat_j = [att.features.index(f) for f in feats_9c]


def _rows_desc(idx):
    "idx sorted by score descending — a DISPLAY order, independent of how §9b picked the claims."
    order = idx[np.argsort(-s[idx])]
    return order, s[order], row_ids[order]


above_order, above_scores, above_ids = _rows_desc(near_above_idx)
below_order, below_scores, below_ids = _rows_desc(near_below_idx)

if len(above_order) == 0 or len(below_order) == 0:
    print(f"one side is empty (above={len(above_order)}, below={len(below_order)}) — "
          f"nothing to plot side by side.")
else:
    above_phi = att.phi[np.ix_(above_order, feat_j)]
    below_phi = att.phi[np.ix_(below_order, feat_j)]
    vmax = float(np.abs(np.concatenate([above_phi, below_phi])).max())

    fig, axes = plt.subplots(
        1, 2, figsize=(sk.FIG_W + 2.6, 0.16 * max(len(above_order), len(below_order)) + 2.4))

    for ax, phi_mat, scores_, label in (
        (axes[0], above_phi, above_scores, f"above τ  (n={len(above_order)})"),
        (axes[1], below_phi, below_scores, f"below τ  (n={len(below_order)})"),
    ):
        im = ax.imshow(phi_mat, aspect="auto", cmap=sk.CMAP, vmin=-vmax, vmax=vmax)
        ax.set_xticks(np.arange(len(feats_9c)))
        ax.set_xticklabels(feats_9c, rotation=90, fontsize=7)
        ax.set_yticks(np.arange(len(scores_)))
        ax.set_yticklabels([f"{v:.4f}" for v in scores_], fontsize=5.5)
        ax.set_title(label, fontsize=9)
    axes[0].set_ylabel("claim score (descending)")

    fig.colorbar(im, ax=list(axes), fraction=0.025, pad=0.02, label="SHAP value  (log-odds)")
    fig.suptitle(f"{VERSION} — individual near-τ claims, φ by feature "
                 f"({len(feats_9c)} features, {ACTIVE_BACKEND})")
    if sk.figstyle:
        sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_09d_near_tau_individual_heatmap")
    plt.show()

## 10 · Inside the fast-track region — mean |SHAP| by score band

Above the cutoff every claim gets the same action, so the variation that remains is *how deep* into
the region it sits. Pooling τ+0.001 with 0.999 hides exactly the thing worth seeing: whether the
model uses the same reasons throughout, or switches to a different set for the claims it is most
certain about. Bands are quantiles **within** the above-τ population, with below-τ kept as the
reference bar.

A feature whose mass grows monotonically into the deepest band is what the fast-track rule is
effectively keyed on — and therefore the feature whose forced labels feed the next version.

In [ ]:
bands = sk.score_bands(s, TAU, quantiles=(0.5, 0.9, 0.99))
print(bands.value_counts().to_frame("claims"))

fig, band_table = sk.plot_band_bars(att, bands, top_n=12,
                                    title=f"{VERSION} — mean |SHAP| by score band ({ACTIVE_BACKEND})")
if sk.figstyle:
    sk.figstyle.save(fig, f"{VERSION}_{ACTIVE_BACKEND}_10a_band_bars")
plt.show()
display(band_table.round(4))

## 11 · Save the attributions — both backends

`ACTIVE_BACKEND`'s result is written to the **canonical** name (exactly the layout
`src/scoring/attribute.py` produces: `<v>_attributions_<split>.parquet`), so this notebook and
the headless CLI stay interchangeable and `00_shap_attribution.ipynb` can read it without being
told which backend it is. The **other** backend is written alongside it with `_<backend>` in the
filename — a sibling artefact this notebook's own within-run comparison produces, not something
the cross-version notebook looks for. Each gets its own sidecar meta (backend, row count, this
version's hyperparameters) — the record without which two attribution files must not be
compared.

In [ ]:
out_path = config.path("attributions", VERSION, SOURCE, split=SPLIT)
out_path.parent.mkdir(parents=True, exist_ok=True)

def save_attribution(a, backend_name, path):
    """Write phi + sidecar meta for one backend's Attribution `a` to `path`."""
    frame_out = a.frame(id_values=row_ids, id_col=ID_COL)
    try:
        frame_out.to_parquet(path, index=False)
    except Exception as exc:                   # env-v1's old pandas/pyarrow may not write parquet
        path = path.with_suffix(".csv")
        frame_out.to_csv(path, index=False)
        print(f"parquet write failed in this env ({type(exc).__name__}: {exc})")
        print(f"-> wrote CSV instead: {path}")
        print("   convert it to parquet in the analysis .venv before 00_shap_attribution.ipynb reads it.")
        print("   NOTE: v1_csv_to_parquet.py does NOT cover detection/shap/ — it walks")
        print("   processed_inputs/targets/scores only. Convert this file by hand:")
        print(f"   pd.read_csv(...).to_parquet('{path.with_suffix('.parquet').name}', index=False)")

    meta = {
        "version": VERSION,
        "split": SPLIT,      # same field attribute.py writes — concentration is unreadable without it
        "model_path": str(model_path),
        "features_provenance": provenance,
        "estimator": type(est).__name__,
        "estimator_params": {k: (v if isinstance(v, (int, float, bool, str)) or v is None else str(v))
                             for k, v in sorted(est.get_params().items())},
        "backend": a.backend,
        "perturbation": a.perturbation,
        "model_output": "raw",
        "note": a.note,
        "n_rows": int(a.phi.shape[0]),
        "n_features": int(a.phi.shape[1]),
        "feature_names": a.features,
        "background_n": int(len(background)) if backend_name == "interventional" else 0,
        "explain_ids_file": None,
        "background_ids_file": None,
        "seed": SEED,
        "tau_used": float(TAU),
        "tau_note": tau_note,   # says HARDCODED for v2 right now — see §3
        "base_value": float(np.mean(a.base)),
        "env": ENV,
    }
    meta_path = path.with_name(path.stem + "_meta.json")
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"[{backend_name}] wrote {path}\n{'':10}{meta_path}")

# ACTIVE_BACKEND -> the canonical name (what attribute.py writes and 00_shap_attribution.ipynb
# reads without knowing which backend it is). The other backend -> the same name with a
# suffix, so neither run overwrites the other. That suffix is attribute_all.py's --out-suffix
# spelling, NOT the backend key: 00_shap_attribution.ipynb's RUNS["path_dependent"] reads
# "_native", so a file named "_tree_path_dependent" would never be found.
OUT_SUFFIX = {"interventional": "_shap", "tree_path_dependent": "_native"}

for name, a in att_by_backend.items():
    p = (out_path if name == ACTIVE_BACKEND
         else out_path.with_name(f"{out_path.stem}{OUT_SUFFIX[name]}{out_path.suffix}"))
    save_attribution(a, name, p)

## Notes — what would make the figures above wrong

- **Wrong kernel.** §0 raises if this env's xgboost is not the release `config` says the pickle was
  serialised with. Do not relax that check: a mismatched load produces figures, not errors.
- **`explain_ids_file: null`.** This notebook samples its own rows, which is right for a
  single-version analysis. Before comparing versions, either re-run the sampling through
  `src/scoring/attribute_all.py` (one shared claim set) or state the case-mix caveat.
- **Backend.** §4 now computes and saves both `interventional` and `tree_path_dependent` every
  run (§11 writes one file per backend); §5–§10's plots only show whichever one `ACTIVE_BACKEND`
  names. `interventional` is measured against the fixed `N_BACKGROUND` sample; `tree_path_dependent`
  against the model's own training distribution. Fine to report either within one version; not
  mixable across versions, and not mixable with each other inside one figure.
- **τ is approximate for v1 and v2.** v1's real rule needs the mobility column per row; v2's needs
  the claim's date to pick the regime. v2's TAU is currently **hardcoded to 0.872** (§3) rather
  than derived from `rule["regimes"]` — a stand-in, not the real rule, which alternates across 5
  regimes over time. §3 prints which simplification it made. For anything that turns on the exact
  treated set, use `src/threshold.py:apply`, which dispatches on the rule shape.
- **Never collapse raw column names to compare with another version.** `make_FORD` and `make_BMW` are the
  measurement. Cross-version correspondence comes only from the hand-confirmed mapping
  (`features/check_overlap.py` → `features/feature_overlap.json`) — never from name equality, and
  nowhere near these numbers.